# ECG-FM trên PTB-XL ở 250 Hz

Notebook này áp dụng pipeline tiền xử lý của MIMIC-IV-ECG cho PTB-XL: chuẩn hóa thứ tự 12 chuyển đạo, resample về 250 Hz, chuẩn hóa từng ECG và cắt các đoạn không chồng lấp 5 giây. Mô hình ECG-FM được khởi tạo lại ngẫu nhiên và huấn luyện toàn bộ từ đầu trên 5 nhóm chẩn đoán PTB-XL (`NORM`, `MI`, `STTC`, `CD`, `HYP`). Split sử dụng `strat_fold`: train 1--8, validation 9, test 10.

Các checkpoint được lưu tại `outputs/ptbxl_ecg_fm_250hz/{best_checkpoint,last_checkpoint}.pt`.

In [1]:
from pathlib import Path
import ast
import copy
import random
from itertools import chain

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
import wfdb

from ecg_transform.inp import ECGInput, ECGInputSchema
from ecg_transform.sample import ECGMetadata, ECGSample
from ecg_transform.t.base import ECGTransform
from ecg_transform.t.common import HandleConstantLeads, LinearResample, ReorderLeads
from ecg_transform.t.cut import SegmentNonoverlapping
from ecg_transform.t.scale import Standardize
from fairseq_signals.models import build_model_from_checkpoint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PTBXL_ROOT = next((p for p in (ROOT / 'data' / 'ptb-xl').iterdir() if (p / 'ptbxl_database.csv').exists()), None)
if PTBXL_ROOT is None:
    raise FileNotFoundError('Không tìm thấy thư mục PTB-XL đã giải nén trong data/ptb-xl.')
CHECKPOINT_TEMPLATE = ROOT / 'ckpts' / 'mimic_iv_ecg_finetuned.pt'
if not CHECKPOINT_TEMPLATE.is_file():
    raise FileNotFoundError(f'Không tìm thấy checkpoint kiến trúc: {CHECKPOINT_TEMPLATE}')
OUTPUT_DIR = ROOT / 'outputs' / 'ptbxl_ecg_fm_250hz'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAMPLE_RATE = 250
SEGMENT_LENGTH = 5 * SAMPLE_RATE
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 30
LR = 1e-4
print({'device': str(DEVICE), 'ptbxl_root': str(PTBXL_ROOT), 'output_dir': str(OUTPUT_DIR)})

/home/nguyenhuynh/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cuda', 'ptbxl_root': '/home/hdd/Vlong/ecg-fm/data/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1', 'output_dir': '/home/hdd/Vlong/ecg-fm/outputs/ptbxl_ecg_fm_250hz'}


In [2]:
db = pd.read_csv(PTBXL_ROOT / 'ptbxl_database.csv')
scp = pd.read_csv(PTBXL_ROOT / 'scp_statements.csv', index_col=0)
diagnostic_classes = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
diagnostic_map = scp['diagnostic_class'].dropna().to_dict()

def diagnostic_labels(value):
    codes = ast.literal_eval(value) if isinstance(value, str) else {}
    return [int(any(diagnostic_map.get(code) == cls for code in codes)) for cls in diagnostic_classes]

db['labels'] = db['scp_codes'].map(diagnostic_labels)
label_matrix = np.asarray(db['labels'].tolist(), dtype=np.float32)
db = db.assign(**{name: label_matrix[:, i] for i, name in enumerate(diagnostic_classes)})
split_names = {**{fold: 'train' for fold in range(1, 9)}, 9: 'valid', 10: 'test'}
db['split'] = db['strat_fold'].map(split_names)
db = db[db['split'].notna()].reset_index(drop=True)
print(db['split'].value_counts().sort_index())
print('positive prevalence:', db.groupby('split')[diagnostic_classes].mean().round(4))

split
test      2203
train    17441
valid     2193
Name: count, dtype: int64
positive prevalence:          NORM      MI    STTC      CD     HYP
split                                        
test   0.4376  0.2510  0.2374  0.2261  0.1194
train  0.4362  0.2516  0.2404  0.2243  0.1216
valid  0.4364  0.2481  0.2435  0.2266  0.1236


In [3]:
class PTBXLFMDataset(Dataset):
    def __init__(self, frame, transforms, schema):
        self.frame, self.transforms, self.schema = frame.reset_index(drop=True), transforms, schema

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        record = str(PTBXL_ROOT / row.filename_hr)
        signal, fields = wfdb.rdsamp(record)
        signal = signal.T.astype(np.float32, copy=False)
        lead_map = {'AVR': 'aVR', 'AVL': 'aVL', 'AVF': 'aVF'}
        lead_names = [lead_map.get(name, name) for name in fields['sig_name']]
        metadata = ECGMetadata(sample_rate=int(fields['fs']), num_samples=signal.shape[1], lead_names=lead_names, unit=None, input_start=0, input_end=signal.shape[1])
        metadata.file = record
        sample = ECGSample(ECGInput(signal, metadata), self.schema, self.transforms)
        target = torch.tensor(row[diagnostic_classes].to_numpy(dtype=np.float32))
        return torch.from_numpy(sample.out).float(), record, target

def collate_fn(items):
    sources = torch.cat([item[0] for item in items])
    records = list(chain.from_iterable([[item[1]] * item[0].shape[0] for item in items]))
    targets = torch.cat([item[2].repeat(item[0].shape[0], 1) for item in items])
    return sources, records, targets

schema = ECGInputSchema(sample_rate=SAMPLE_RATE, expected_lead_order=LEADS, required_num_samples=SEGMENT_LENGTH)
transforms = [ReorderLeads(expected_order=LEADS, missing_lead_strategy='raise'), LinearResample(desired_sample_rate=SAMPLE_RATE), HandleConstantLeads(strategy='zero'), Standardize(), SegmentNonoverlapping(segment_length=SEGMENT_LENGTH)]
loaders = {}
for split in ('train', 'valid', 'test'):
    dataset = PTBXLFMDataset(db[db['split'] == split], transforms, schema)
    loaders[split] = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=split == 'train', num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', collate_fn=collate_fn)
print({split: len(loader.dataset) for split, loader in loaders.items()})

{'train': 17441, 'valid': 2193, 'test': 2203}


In [4]:
# Dùng checkpoint chỉ để dựng đúng ECG-FM; toàn bộ tham số được reset nên không transfer learning.
model = build_model_from_checkpoint(checkpoint_path=str(CHECKPOINT_TEMPLATE))
classifier_features = model.proj.in_features
for module in model.modules():
    reset = getattr(module, 'reset_parameters', None)
    if callable(reset):
        reset()
model.proj = torch.nn.Linear(classifier_features, len(diagnostic_classes))
torch.nn.init.xavier_uniform_(model.proj.weight)
torch.nn.init.constant_(model.proj.bias, 0.0)
model.to(DEVICE)

train_frame = db[db['split'] == 'train']
positive = train_frame[diagnostic_classes].sum().to_numpy(dtype=np.float32)
negative = len(train_frame) - positive
pos_weight = torch.tensor(negative / np.maximum(positive, 1), dtype=torch.float32, device=DEVICE)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), weight_decay=1e-4)
print('train-from-scratch:', sum(p.numel() for p in model.parameters()), 'parameters')

train-from-scratch: 90377093 parameters


In [5]:
def record_level(values, records, reducer='mean'):
    frame = pd.DataFrame(values, index=records, columns=diagnostic_classes)
    return frame.groupby(level=0).agg(reducer).sort_index()

def metrics(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(np.int32)
    auroc = [roc_auc_score(y_true[:, i], y_score[:, i]) for i in range(y_true.shape[1]) if np.unique(y_true[:, i]).size == 2]
    auprc = [average_precision_score(y_true[:, i], y_score[:, i]) for i in range(y_true.shape[1]) if np.unique(y_true[:, i]).size == 2]
    f1, recall, specificity = [], [], []
    for i in range(y_true.shape[1]):
        tp = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 1)); tn = np.sum((y_true[:, i] == 0) & (y_pred[:, i] == 0))
        fp = np.sum((y_true[:, i] == 0) & (y_pred[:, i] == 1)); fn = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 0))
        f1.append(2 * tp / max(2 * tp + fp + fn, 1)); recall.append(tp / max(tp + fn, 1)); specificity.append(tn / max(tn + fp, 1))
    return {'MACRO-AUROC': float(np.mean(auroc)), 'macro-AUPRC': float(np.mean(auprc)), 'Macro-F1': float(np.mean(f1)), 'Macro-Recall': float(np.mean(recall)), 'Macro-specificity': float(np.mean(specificity))}

def evaluate(split):
    model.eval(); losses, scores, targets, records = [], [], [], []
    with torch.inference_mode():
        for source, batch_records, target in loaders[split]:
            output = model(source=source.to(DEVICE))['out']
            losses.append(criterion(output, target.to(DEVICE)).item())
            scores.append(torch.sigmoid(output).cpu().numpy()); targets.append(target.numpy()); records.extend(batch_records)
    score = record_level(np.concatenate(scores), records); target = record_level(np.concatenate(targets), records, reducer='max')
    return float(np.mean(losses)), metrics(target.to_numpy(), score.to_numpy()), target, score

In [ ]:
history, best_value = [], -np.inf
best_path, last_path = OUTPUT_DIR / 'best_checkpoint.pt', OUTPUT_DIR / 'last_checkpoint.pt'
for epoch in range(1, EPOCHS + 1):
    model.train(); train_losses = []
    for source, _, target in loaders['train']:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(source=source.to(DEVICE))['out'], target.to(DEVICE))
        loss.backward(); optimizer.step(); train_losses.append(loss.item())
    valid_loss, valid_metrics, _, _ = evaluate('valid')
    row = {'epoch': epoch, 'train_loss': float(np.mean(train_losses)), 'valid_loss': valid_loss, **{f'valid_{k}': v for k, v in valid_metrics.items()}}
    history.append(row)
    checkpoint = {'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'history': history, 'labels': diagnostic_classes, 'sample_rate': SAMPLE_RATE}
    torch.save(checkpoint, last_path)
    if valid_metrics['macro-AUPRC'] > best_value:
        best_value = valid_metrics['macro-AUPRC']; torch.save(checkpoint, best_path)
    print(row)

pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
best = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best['model_state_dict'])
test_loss, test_metrics, test_targets, test_scores = evaluate('test')
print('TEST LOSS:', test_loss)
print(pd.Series(test_metrics, name='test'))
test_scores.to_csv(OUTPUT_DIR / 'test_scores.csv')
test_targets.to_csv(OUTPUT_DIR / 'test_targets.csv')
assert best_path.is_file() and last_path.is_file()

{'epoch': 1, 'train_loss': 0.8838845496321772, 'valid_loss': 0.9125611751839735, 'valid_MACRO-AUROC': 0.8488761717417541, 'valid_macro-AUPRC': 0.6425551003743358, 'valid_Macro-F1': 0.6137118674973465, 'valid_Macro-Recall': 0.7364584846617177, 'valid_Macro-specificity': 0.7892445553606023}
{'epoch': 2, 'train_loss': 0.7454615844475044, 'valid_loss': 0.7373824767444445, 'valid_MACRO-AUROC': 0.8575029017221484, 'valid_macro-AUPRC': 0.6506126881325647, 'valid_Macro-F1': 0.6045927254924642, 'valid_Macro-Recall': 0.8321712178911633, 'valid_Macro-specificity': 0.7210581977526933}
{'epoch': 3, 'train_loss': 0.7553837624115821, 'valid_loss': 0.957572643307672, 'valid_MACRO-AUROC': 0.7221973958808485, 'valid_macro-AUPRC': 0.46574803558698463, 'valid_Macro-F1': 0.49163155811483883, 'valid_Macro-Recall': 0.6721202939183623, 'valid_Macro-specificity': 0.6096116862887148}
{'epoch': 4, 'train_loss': 0.909309311118318, 'valid_loss': 0.8997695385545924, 'valid_MACRO-AUROC': 0.7554823528031683, 'valid_m